In [1]:
import os

from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torch
import numpy as np
import cv2

from baselines.ViT.ViT_LRP import SimpleVisionTransformer as vit_LRP
# To verify that the logits are the same as the unconverted model:
# from simple_vit import SimpleVisionTransformer as vit_LRP

In [2]:
def convert(state):
    for k in list(state.keys()):
    	l = k.split('.')
    	if l[-2:] == ['self_attention', 'qkv_w']:
            l[-1] = 'qkv.weight'
            state[k] = state[k].flatten(end_dim=1)
    	elif l[-3:-1] == ['self_attention', 'out']:
            l[-2] = 'proj'
    	state['.'.join(l)] = state.pop(k)
    return state

In [3]:
hidden_dim = 384
input_resolution = 224
vit = vit_LRP(
	image_size=input_resolution,
	patch_size=16,
	num_layers=12,
	num_heads=6,
	hidden_dim=hidden_dim,
	mlp_dim=hidden_dim * 4,
	representation_size=hidden_dim,
)

In [4]:
ROOT = "/media/jason-chou/T31/imagenet-runs/logs"
run = f"{300}ep-Scion-2x-wd"
p = os.path.join(ROOT, run)

best_path = os.path.join(p, 'checkpoints/model_best.pth.tar')
ckpt = torch.load(best_path, weights_only=True)
state = ckpt['state_dict']
state = convert(state) # Comment this out for unconverted models

vit.load_state_dict({k[len('module.'):]: v for k, v in state.items()})

<All keys matched successfully>

In [5]:
import torchvision.datasets as datasets
from torchvision.transforms import v2

data = "/data/ImageNet/"

value_range = v2.Normalize(
    mean=[0.5] * 3,
    std=[0.5] * 3
)

transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(256),
    v2.CenterCrop(input_resolution),
    v2.ToDtype(torch.float32, scale=True),
    value_range,
])

val_dataset = datasets.ImageNet(
    data,
    split='val',
    transform=transform
)

In [6]:
device = torch.device("cuda")
vit = vit.to(device)

In [7]:
# To verify that the logits are the same as the unconverted model:

# image, target = next(iter(val_dataset))
# image = image.cuda()
# target = torch.Tensor([target]).type(torch.LongTensor).cuda()
# vit(image.unsqueeze(0), 1.0, target, target)[0]

In [8]:
# image, target = next(iter(val_dataset))
# vit(image.unsqueeze(0).cuda())

In [9]:
from scipy.stats import entropy
from tqdm.notebook import tqdm
from baselines.ViT.ViT_explanation_generator import LRP

def attn_rel(p, gen, val_dataset):
    best_path = os.path.join(p, 'checkpoints/model_best.pth.tar')
    ckpt = torch.load(best_path, weights_only=True)
    state = convert(ckpt['state_dict'])
    vit = gen.model
    try:
        vit.load_state_dict(state)
    except:
        vit.load_state_dict({k[len('module.'):]: v for k, v in state.items()})
    result = []
    for image, target in tqdm(val_dataset):
        lrp = gen.generate_LRP(image.unsqueeze(0).cuda(), index=target, method="last_layer")
        result.append(entropy(lrp.detach().cpu().numpy()))
    return np.array(result)

gen = LRP(vit)
entropy_file = 'attn_entropy.npy'

In [10]:
scion_runs = []
for postfix in ['', '-0', '-1', '-2']:
    for ep in [30, 60, 90, 150, 300]:
        scion_runs.append(f"{ep}ep-Scion" + postfix)
        for f in [2, 3, 4]:
            scion_runs.append(f"{ep}ep-Scion-{f}x-wd" + postfix)

In [11]:
def attn_rel_sweep(root, gen, val_dataset):
    entropy_file = 'attn_entropy.npy'
    for run in sorted(os.listdir(root)):
        if os.path.isdir(os.path.join(ROOT, run)) and (run in scion_runs or ('Scion' in run and 'lower-scale' in run)):
            p = os.path.join(root, run)
            if os.path.isdir(p):
                if os.path.exists(entropy_path := os.path.join(p, entropy_file)):
                    print(p + ' already done!')
                    continue
                else:
                    print(p)
                    ent = attn_rel(p, gen, val_dataset)
                    filename = entropy_file.split('.')[0]
                    np.save(os.path.join(p, filename), ent)    

attn_rel_sweep(ROOT, gen, val_dataset)

/media/jason-chou/T31/imagenet-runs/logs/150ep-Scion


  0%|          | 0/50000 [00:00<?, ?it/s]

/media/jason-chou/T31/imagenet-runs/logs/150ep-Scion-1


  0%|          | 0/50000 [00:00<?, ?it/s]

/media/jason-chou/T31/imagenet-runs/logs/150ep-Scion-2


...

  0%|          | 0/50000 [00:00<?, ?it/s]

/media/jason-chou/T31/imagenet-runs/logs/90ep-Scion-4x-wd-1


  0%|          | 0/50000 [00:00<?, ?it/s]

/media/jason-chou/T31/imagenet-runs/logs/90ep-Scion-4x-wd-2


  0%|          | 0/50000 [00:00<?, ?it/s]